# Week 4: Statistical Analysis and Predictive Modeling in Healthcare
## Coronary Artery Disease (UCI Heart Disease Databases)

**Project continuity:** This notebook follows the Week 2 data collection/quality-assurance pipeline and Week 3 visualization work. It expects `qa_output/heart_disease_clean.csv` created by Week 2, or a CSV path supplied below.

**Objective:** Build a reproducible, educational framework for exploring clinical trends and evaluating candidate models that predict the dataset's binary heart-disease label.

> **Important:** This is a retrospective educational exercise using a historical public dataset. It is not a diagnostic device, does not establish causation, and must not be used for patient care. Metrics from this dataset do not demonstrate real-world clinical safety or generalisability.


## 1. Research question and analysis plan

**Primary question:** How well can routinely recorded demographic and clinical variables distinguish records labelled as having any heart disease (`target = 1`) from records labelled as having no heart disease (`target = 0`)?

**Secondary questions**
- How do age, sex, chest-pain category, exercise-induced angina and ST depression vary by outcome?
- Which candidate model performs best on held-out records?
- Do performance and error rates differ across available subgroups or source sites?

**Unit of analysis:** one patient record. **Outcome:** binary `target`, where 1 indicates any disease and 0 indicates no disease according to the dataset's original coding. **Intended use:** exploratory research and model-development practice only.

### Planned methods
- Descriptive statistics: mean/SD or median/IQR, counts/percentages, missingness summaries.
- Association tests: chi-square for categorical variables; t-test or Mann–Whitney U for continuous variables, with effect sizes and confidence intervals preferred over p-values alone.
- Prediction: regularised logistic regression as an interpretable baseline and random forest as a non-linear comparator.
- Validation: stratified train/test split plus stratified cross-validation on training data; report sensitivity, specificity, precision, F1, ROC-AUC, PR-AUC, confusion matrix and calibration.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, brier_score_loss
)
from sklearn.calibration import CalibrationDisplay
from IPython.display import display

RANDOM_STATE = 42
DATA_PATH = Path("qa_output/heart_disease_clean.csv")
OUTPUT_DIR = Path("week4_output")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Working directory:", Path.cwd())
print("Expected dataset:", DATA_PATH.resolve())


## 2. Load data and verify the target

Run Week 2 first to generate the cleaned dataset. If your file is stored elsewhere, update `DATA_PATH`. This notebook intentionally does not silently generate clinical data when the real file is missing.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Run Week2_QA_Pipeline.ipynb first, "
        "or update DATA_PATH to the location of heart_disease_clean.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Rows, columns:", df.shape)
display(df.head())
print("\\nColumns:")
print(df.columns.tolist())

if "target" not in df.columns:
    if "num" in df.columns:
        df["target"] = (pd.to_numeric(df["num"], errors="coerce") > 0).astype(int)
    else:
        raise ValueError("No target column found; expected 'target' or original 'num'.")

df["target"] = pd.to_numeric(df["target"], errors="coerce")
df = df[df["target"].isin([0, 1])].copy()
df["target"] = df["target"].astype(int)
print("\\nTarget counts:")
display(df["target"].value_counts().rename(index={0:"No disease",1:"Disease"}).to_frame("records"))


## 3. Data audit and segmentation

Before modelling, inspect row counts, duplicate records, missingness, outcome balance, and site composition. Site can be important for validation because measurement practices and patient mix may differ between institutions. Do not assume site differences represent clinical differences alone.


In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "unique_n": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)
display(audit)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Outcome prevalence (%):", round(100*df["target"].mean(), 2))

if "site" in df.columns:
    site_summary = df.groupby("site", dropna=False).agg(
        records=("target","size"),
        positive_cases=("target","sum"),
        prevalence=("target","mean")
    )
    site_summary["prevalence_pct"] = 100*site_summary.pop("prevalence")
    display(site_summary.round(3))


## 4. Descriptive statistics and statistical comparisons

These comparisons are exploratory. A p-value does not measure clinical importance and multiple comparisons can produce false positives. Report effect sizes and uncertainty when preparing a formal analysis.


In [ ]:
# Identify common variables when present; adapt this list to the actual dataset.
numeric_candidates = [c for c in ["age","trestbps","chol","thalach","oldpeak"] if c in df.columns]
categorical_candidates = [c for c in ["sex","cp","fbs","restecg","exang","slope","ca","thal"] if c in df.columns]

desc_rows = []
for col in numeric_candidates:
    x0 = pd.to_numeric(df.loc[df.target==0, col], errors="coerce").dropna()
    x1 = pd.to_numeric(df.loc[df.target==1, col], errors="coerce").dropna()
    desc_rows.append({
        "variable": col, "no_disease_n": len(x0), "disease_n": len(x1),
        "no_disease_median": x0.median() if len(x0) else np.nan,
        "disease_median": x1.median() if len(x1) else np.nan,
        "no_disease_mean": x0.mean() if len(x0) else np.nan,
        "disease_mean": x1.mean() if len(x1) else np.nan,
        "mann_whitney_p": stats.mannwhitneyu(x0, x1, alternative="two-sided").pvalue if len(x0) and len(x1) else np.nan
    })
numeric_summary = pd.DataFrame(desc_rows)
display(numeric_summary.round(4))
numeric_summary.to_csv(OUTPUT_DIR/"numeric_group_comparisons.csv", index=False)

cat_rows = []
for col in categorical_candidates:
    tab = pd.crosstab(df[col], df["target"])
    if tab.shape[0] > 1 and tab.shape[1] > 1:
        chi2, p, dof, expected = stats.chi2_contingency(tab)
        cat_rows.append({"variable":col, "chi_square":chi2, "degrees_freedom":dof,
                         "p_value":p, "minimum_expected_count":expected.min()})
        print(f"\\n{col}: contingency table")
        display(tab)
cat_summary = pd.DataFrame(cat_rows)
display(cat_summary.round(4))
cat_summary.to_csv(OUTPUT_DIR/"categorical_association_tests.csv", index=False)


In [ ]:
# Visual summaries: outcome balance and selected numeric distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["target"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_xticks([0,1], ["No disease (0)", "Disease (1)"], rotation=0)
axes[0].set_ylabel("Record count")
axes[0].set_title("Outcome distribution")
if "age" in df.columns:
    for label, group in df.groupby("target"):
        axes[1].hist(pd.to_numeric(group["age"], errors="coerce").dropna(),
                     bins=12, alpha=.55, label="Disease" if label==1 else "No disease")
    axes[1].set_xlabel("Age (years)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Age distribution by outcome")
    axes[1].legend()
else:
    axes[1].axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"descriptive_distributions.png", bbox_inches="tight")
plt.show()


## 5. Define predictors and prevent target leakage

The original `num` field is the disease severity/diagnosis grade used to derive the binary target. It must **not** be a predictor. Likewise, exclude `target`, `num_grade`, post-outcome variables, and any duplicate target-derived fields. Missingness indicator columns created by Week 2 may be used only if they would be available at the intended prediction time and their meaning is documented; for this baseline they are excluded.

The categorical list below follows the clinical meaning of the UCI variables. Confirm the actual encoding and site-specific definitions before any real deployment.


In [ ]:
# Candidate predictor set; use only columns present in the loaded data.
numeric_features = [c for c in ["age","trestbps","chol","thalach","oldpeak"] if c in df.columns]
categorical_features = [c for c in ["sex","cp","fbs","restecg","exang","slope","ca","thal"] if c in df.columns]
feature_cols = numeric_features + categorical_features

# Optional source site is reserved for subgroup analysis / validation, not a predictor in this baseline.
excluded = {"target","num","num_grade","disease","record_id","site"}
feature_cols = [c for c in feature_cols if c not in excluded]
if len(feature_cols) == 0:
    raise ValueError("No expected clinical predictor columns found. Check the dataset schema.")
X = df[feature_cols].copy()
y = df["target"].astype(int).copy()

# Convert feature columns to numeric where appropriate; invalid values become missing and are imputed inside the pipeline.
for c in numeric_features + categorical_features:
    X[c] = pd.to_numeric(X[c], errors="coerce")

print("Predictors:", feature_cols)
print("Excluded outcome/identifier fields:", sorted(excluded))
print("Class counts:", y.value_counts().to_dict())


## 6. Split the data before fitting any preprocessing

A stratified hold-out test set is reserved for final evaluation. Imputation, scaling and encoding are performed inside an sklearn `Pipeline`, so cross-validation learns preprocessing only from each training fold. If multiple rows per patient exist, split by patient identifier; if evaluating transportability across institutions, use a site-held-out design instead of a random split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training records:", len(X_train), "| Test records:", len(X_test))
print("Training positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 7. Candidate models and cross-validation

Logistic regression is a transparent baseline and provides a useful comparison. Random forest can capture nonlinear relationships and interactions but is less directly interpretable. Model selection is based on training-set cross-validation only; the test set remains untouched until the final evaluation. With small datasets, fold-level estimates can be unstable, so show variability and avoid overclaiming.


In [ ]:
models = {
    "Logistic regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random forest": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc":"roc_auc", "average_precision":"average_precision",
           "balanced_accuracy":"balanced_accuracy", "recall":"recall", "precision":"precision"}
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    row = {"model":name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = vals.mean()
        row[f"{metric}_sd"] = vals.std(ddof=1)
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows).sort_values("roc_auc_mean", ascending=False)
display(cv_results.round(3))
cv_results.to_csv(OUTPUT_DIR/"cross_validation_model_comparison.csv", index=False)

best_name = cv_results.iloc[0]["model"]
best_model = models[best_name]
print("Selected using training CV ROC-AUC:", best_name)
best_model.fit(X_train, y_train)


## 8. Final hold-out evaluation

Report multiple metrics because no single metric fully captures clinical utility. Sensitivity/recall measures the share of positive cases detected; specificity measures the share of negative cases correctly identified. Precision depends on prevalence. ROC-AUC measures ranking, while PR-AUC is especially informative when the positive class is uncommon. The default 0.50 threshold is only a reference point—not a clinically approved operating threshold.


In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
specificity = tn/(tn+fp) if (tn+fp) else np.nan
metrics = {
    "selected_model": best_name, "threshold_reference_only": threshold,
    "test_n": len(y_test), "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc_average_precision": average_precision_score(y_test, y_prob),
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": (recall_score(y_test,y_pred) + specificity)/2,
    "sensitivity_recall": recall_score(y_test, y_pred, zero_division=0),
    "specificity": specificity,
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "brier_score": brier_score_loss(y_test, y_prob),
    "true_negative": int(tn), "false_positive": int(fp),
    "false_negative": int(fn), "true_positive": int(tp)
}
display(pd.DataFrame([metrics]).T.rename(columns={0:"value"}))
print(classification_report(y_test, y_pred, target_names=["No disease","Disease"], zero_division=0))
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR/"holdout_metrics.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1])
CalibrationDisplay.from_predictions(y_test, y_prob, n_bins=5, strategy="quantile", ax=axes[2])
axes[0].set_title("ROC curve")
axes[1].set_title("Precision–recall curve")
axes[2].set_title("Calibration (small sample: interpret cautiously)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"holdout_evaluation.png", bbox_inches="tight")
plt.show()


## 9. Subgroup performance and fairness checks

Where sample sizes permit, compare sensitivity, specificity, and AUC across sex, age bands and source sites. These estimates can be noisy, especially in small subgroups. Suppress or clearly qualify small cells, use confidence intervals, and investigate differences rather than treating a single metric as proof of bias or fairness. A site-level comparison is not a substitute for external validation.


In [ ]:
test_results = X_test.copy()
test_results["y_true"] = y_test
test_results["y_prob"] = y_prob
test_results["y_pred"] = y_pred
# Add subgroup columns from original records using the preserved row index.
for col in ["sex","site","age"]:
    if col in df.columns:
        test_results[col] = df.loc[X_test.index, col]

if "age" in test_results.columns:
    test_results["age_band"] = pd.cut(
        pd.to_numeric(test_results["age"], errors="coerce"),
        bins=[0,39,49,59,69,120],
        labels=["<=39","40-49","50-59","60-69","70+"],
        include_lowest=True
    )

def subgroup_metrics(g):
    n = len(g)
    positives = int(g.y_true.sum())
    negatives = int((g.y_true==0).sum())
    sens = ((g.y_pred==1)&(g.y_true==1)).sum()/positives if positives else np.nan
    spec = ((g.y_pred==0)&(g.y_true==0)).sum()/negatives if negatives else np.nan
    auc = roc_auc_score(g.y_true, g.y_prob) if g.y_true.nunique()==2 else np.nan
    return pd.Series({"n":n,"positive_n":positives,"sensitivity":sens,"specificity":spec,"roc_auc":auc})

fairness_tables = {}
for col in ["sex","age_band","site"]:
    if col in test_results.columns:
        tbl = test_results.groupby(col, dropna=False, observed=False).apply(subgroup_metrics).reset_index()
        tbl["group_variable"] = col
        fairness_tables[col] = tbl
        print(f"Subgroup performance by {col}")
        display(tbl)
if fairness_tables:
    pd.concat(fairness_tables.values(), ignore_index=True).to_csv(
        OUTPUT_DIR/"subgroup_fairness_metrics.csv", index=False
    )


## 10. Interpretation, limitations and responsible use

### How to interpret findings
- Treat associations as descriptive; observational records cannot establish causation.
- Use confidence intervals and sample counts alongside percentages.
- Review model errors with clinical collaborators; false negatives and false positives have different consequences.
- Select any operating threshold only after defining the clinical use, assessing harms, and conducting decision-curve or utility analysis with stakeholders.

### Main threats to validity
1. **Historical data and limited representativeness:** older, small, institution-specific records may not reflect current populations or practice.
2. **Missing data and imputation:** missingness may depend on clinical workflow or disease severity. Median/mode imputation can understate uncertainty.
3. **Site heterogeneity:** different sites may use different populations, protocols, and coding.
4. **Class imbalance:** accuracy can conceal poor minority-class detection.
5. **Leakage:** diagnosis-derived variables or preprocessing performed before splitting can inflate results.
6. **Small test set:** metrics and subgroup comparisons can have wide uncertainty.
7. **Calibration and prevalence shift:** predicted probabilities may not transfer to a new hospital.
8. **Confounding and proxy variables:** variables such as sex or site can encode structural differences.
9. **Repeated records:** if patients appear more than once, random splitting can leak information across sets.

### Ethical safeguards
- Use only authorised, de-identified data; follow applicable ethics review, privacy and data-protection requirements.
- Document dataset provenance, inclusion/exclusion criteria, transformations, intended use, and model version.
- Involve clinicians, statisticians, data-governance staff and affected communities in defining the use case.
- Evaluate subgroup performance and uncertainty; do not infer fairness from overall accuracy.
- Keep a human decision-maker in the loop; do not use the model as a stand-alone diagnosis or to deny care.
- Provide understandable explanations and a route to challenge or review model-supported decisions.
- Monitor performance, calibration, drift, data quality and harms after deployment; establish rollback criteria.
- Do not deploy without independent external validation, prospective evaluation, clinical utility assessment, security review and governance approval.

## 11. Reproducibility and output files

The notebook saves tables and figures under `week4_output/`:
- `numeric_group_comparisons.csv`
- `categorical_association_tests.csv`
- `cross_validation_model_comparison.csv`
- `holdout_metrics.csv`
- `subgroup_fairness_metrics.csv` (when subgroup fields exist)
- `descriptive_distributions.png`
- `holdout_evaluation.png`

Record Python and package versions, dataset source/version, random seed, split strategy and any manual changes when submitting. Do not report numerical findings until the notebook has been run against the actual cleaned dataset.
